In [12]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

from tensorflow.keras import layers
from tensorflow.keras.datasets import imdb

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.21.0


In [13]:
# Load IMDb dataset
# 0 = Negative
# 1 = Positive

(x_train, y_train), (x_test, y_test) = imdb.load_data(
    num_words=10000
)

print("Training samples:", len(x_train))
print("Testing samples:", len(x_test))

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 287s 16us/step
Training samples: 25000
Testing samples: 25000


In [22]:
word_index = imdb.get_word_index()

# Reverse the word index
reverse_word_index = {
    value: key for key, value in word_index.items()
}

print("Vocabulary loaded successfully!")

Vocabulary loaded successfully!


In [23]:
def decode_review(review):
    words = []

    for number in review:
        word = reverse_word_index.get(number - 3, "?")
        words.append(word)

    return " ".join(words)


# Convert training reviews to text
train_text = []

for review in x_train:
    train_text.append(decode_review(review))


# Convert testing reviews to text
test_text = []

for review in x_test:
    test_text.append(decode_review(review))


print("First review:")
print(train_text[0])

First review:
? this film was just brilliant casting location scenery story direction everyone's really suited the part they played and you could just imagine being there robert ? is an amazing actor and now the same being director ? father came from the same scottish island as myself so i loved the fact there was a real connection with this film the witty remarks throughout the film were great it was just brilliant so much that i bought the film as soon as it was released for ? and would recommend it to everyone to watch and the fly fishing was amazing really cried at the end it was so sad and you know what they say if you cry at a film it must have been good and this definitely was also ? to the two little boy's that played the ? of norman and paul they were just brilliant children are often left out of the ? list i think because the stars that play them all grown up are such a big profile for the whole film but these children are amazing and should be praised for what they have done

In [24]:
vectorizer = layers.TextVectorization(
    max_tokens=10000,
    output_mode="tf_idf"
)

# Learn vocabulary from training text
vectorizer.adapt(train_text)

print("Text Vectorization completed!")
print("Vocabulary size:", len(vectorizer.get_vocabulary()))

Text Vectorization completed!
Vocabulary size: 9907


In [25]:
x_train_vectorized = vectorizer(
    np.array(train_text)
)

x_test_vectorized = vectorizer(
    np.array(test_text)
)

print("Training feature shape:", x_train_vectorized.shape)
print("Testing feature shape:", x_test_vectorized.shape)

Training feature shape: (25000, 9907)
Testing feature shape: (25000, 9907)


In [26]:
# Use first 5,000 training samples for validation
validation_size = 5000

x_val = x_train_vectorized[:validation_size]
y_val = y_train[:validation_size]

x_train_final = x_train_vectorized[validation_size:]
y_train_final = y_train[validation_size:]

print("Training data:", x_train_final.shape)
print("Validation data:", x_val.shape)

Training data: (20000, 9907)
Validation data: (5000, 9907)


In [29]:
model_basic = tf.keras.Sequential([
    tf.keras.Input(shape=(x_train_final.shape[1],)),
    
    tf.keras.layers.Dense(
        64,
        activation="relu"
    ),
    
    tf.keras.layers.Dense(
        1,
        activation="sigmoid"
    )
])

model_basic.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model_basic.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense_8 (Dense)                      │ (None, 64)                  │         634,112 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_9 (Dense)                      │ (None, 1)                   │              65 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 634,177 (2.42 MB)

 Trainable params: 634,177 (2.42 MB)

 Non-trainable params: 0 (0.00 B)

In [30]:
history_basic = model_basic.fit(
    x_train_final,
    y_train_final,
    validation_data=(x_val, y_val),
    epochs=10,
    batch_size=32
)

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - accuracy: 0.8594 - loss: 0.3425 - val_accuracy: 0.8884 - val_loss: 0.2830
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.9635 - loss: 0.1046 - val_accuracy: 0.8826 - val_loss: 0.3541
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 11s 9ms/step - accuracy: 0.9919 - loss: 0.0289 - val_accuracy: 0.8780 - val_loss: 0.4305
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.9987 - loss: 0.0078 - val_accuracy: 0.8812 - val_loss: 0.5377
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.9994 - loss: 0.0025 - val_accuracy: 0.8806 - val_loss: 0.6346
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.9995 - loss: 0.0014 - val_accuracy: 0.8818 - val_loss: 0.6974
Epoch 7/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.9997 - loss: 8.3646e-04 - val_accuracy: 0.8816 - val_loss: 0.7422
Epoch 8/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.9998 - loss: 6.5100e-04 - val_acc

In [33]:
model_dropout = tf.keras.Sequential([
    tf.keras.Input(shape=(x_train_final.shape[1],)),
    
    tf.keras.layers.Dense(
        64,
        activation="relu"
    ),
    
    tf.keras.layers.Dropout(0.5),
    
    tf.keras.layers.Dense(
        1,
        activation="sigmoid"
    )
])

model_dropout.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model_dropout.summary()

Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense_12 (Dense)                     │ (None, 64)                  │         634,112 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_2 (Dropout)                  │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_13 (Dense)                     │ (None, 1)                   │              65 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 634,177 (2.42 MB)

 Trainable params: 634,177 (2.42 MB)

 Non-trainable params: 0 (0.00 B)

In [34]:
history_dropout = model_dropout.fit(
    x_train_final,
    y_train_final,
    validation_data=(x_val, y_val),
    epochs=10,
    batch_size=32
)

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - accuracy: 0.8489 - loss: 0.3659 - val_accuracy: 0.8862 - val_loss: 0.2841
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.9312 - loss: 0.1841 - val_accuracy: 0.8866 - val_loss: 0.3027
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.9625 - loss: 0.1115 - val_accuracy: 0.8850 - val_loss: 0.3267
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.9761 - loss: 0.0714 - val_accuracy: 0.8804 - val_loss: 0.3729
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.9837 - loss: 0.0513 - val_accuracy: 0.8814 - val_loss: 0.4184
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - accuracy: 0.9868 - loss: 0.0408 - val_accuracy: 0.8814 - val_loss: 0.4273
Epoch 7/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.9886 - loss: 0.0338 - val_accuracy: 0.8818 - val_loss: 0.4793
Epoch 8/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - accuracy: 0.9906 - loss: 0.0313 - val_accuracy: 0.

In [38]:
print("Actual input shape:", x_train_final.shape)
model_l2 = tf.keras.Sequential([
    tf.keras.Input(shape=(x_train_final.shape[1],)),
    
    tf.keras.layers.Dense(
        64,
        activation="relu",
        kernel_regularizer=tf.keras.regularizers.l2(0.001)
    ),
    
    tf.keras.layers.Dense(
        1,
        activation="sigmoid"
    )
])

model_l2.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model_l2.summary()

print("Model input shape:", model_l2.input_shape)

Actual input shape: (20000, 9907)


Model: "sequential_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense_18 (Dense)                     │ (None, 64)                  │         634,112 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_19 (Dense)                     │ (None, 1)                   │              65 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 634,177 (2.42 MB)

 Trainable params: 634,177 (2.42 MB)

 Non-trainable params: 0 (0.00 B)

Model input shape: (None, 9907)


In [39]:
history_l2 = model_l2.fit(
    x_train_final,
    y_train_final,
    validation_data=(x_val, y_val),
    epochs=10,
    batch_size=32
)

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 7s 9ms/step - accuracy: 0.8638 - loss: 0.4337 - val_accuracy: 0.8842 - val_loss: 0.4224
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - accuracy: 0.9366 - loss: 0.2928 - val_accuracy: 0.8786 - val_loss: 0.4655
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - accuracy: 0.9582 - loss: 0.2611 - val_accuracy: 0.8718 - val_loss: 0.5057
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - accuracy: 0.9676 - loss: 0.2450 - val_accuracy: 0.8686 - val_loss: 0.5339
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - accuracy: 0.9744 - loss: 0.2384 - val_accuracy: 0.8754 - val_loss: 0.5630
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - accuracy: 0.9748 - loss: 0.2347 - val_accuracy: 0.8696 - val_loss: 0.5795
Epoch 7/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - accuracy: 0.9836 - loss: 0.2132 - val_accuracy: 0.8688 - val_loss: 0.5667
Epoch 8/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - accuracy: 0.9826 - loss: 0.2037 - val_accuracy: 0.

In [40]:
print("MODEL COMPARISON")
print("=" * 70)

# Without Regularization
print("\n1. WITHOUT REGULARIZATION")
print("Training Accuracy :", history_basic.history["accuracy"][-1])
print("Validation Accuracy:", history_basic.history["val_accuracy"][-1])
print("Training Loss     :", history_basic.history["loss"][-1])
print("Validation Loss   :", history_basic.history["val_loss"][-1])

# Dropout
print("\n2. WITH DROPOUT")
print("Training Accuracy :", history_dropout.history["accuracy"][-1])
print("Validation Accuracy:", history_dropout.history["val_accuracy"][-1])
print("Training Loss     :", history_dropout.history["loss"][-1])
print("Validation Loss   :", history_dropout.history["val_loss"][-1])

# L2 Regularization
print("\n3. WITH L2 REGULARIZATION")
print("Training Accuracy :", history_l2.history["accuracy"][-1])
print("Validation Accuracy:", history_l2.history["val_accuracy"][-1])
print("Training Loss     :", history_l2.history["loss"][-1])
print("Validation Loss   :", history_l2.history["val_loss"][-1])

MODEL COMPARISON

1. WITHOUT REGULARIZATION
Training Accuracy : 0.9998499751091003
Validation Accuracy: 0.8794000148773193
Training Loss     : 0.0003646052209660411
Validation Loss   : 0.8665991425514221

2. WITH DROPOUT
Training Accuracy : 0.9922500252723694
Validation Accuracy: 0.8773999810218811
Training Loss     : 0.024296773597598076
Validation Loss   : 0.5591325163841248

3. WITH L2 REGULARIZATION
Training Accuracy : 0.986299991607666
Validation Accuracy: 0.8741999864578247
Training Loss     : 0.20064055919647217
Validation Loss   : 0.5918302536010742


In [41]:
print("COMPARISON")
print("=" * 70)

basic_train_acc = history_basic.history["accuracy"][-1]
basic_val_acc = history_basic.history["val_accuracy"][-1]

dropout_train_acc = history_dropout.history["accuracy"][-1]
dropout_val_acc = history_dropout.history["val_accuracy"][-1]

l2_train_acc = history_l2.history["accuracy"][-1]
l2_val_acc = history_l2.history["val_accuracy"][-1]

print("\nTraining Accuracy:")
print("Without Regularization :", round(basic_train_acc, 4))
print("Dropout                :", round(dropout_train_acc, 4))
print("L2 Regularization      :", round(l2_train_acc, 4))

print("\nValidation Accuracy:")
print("Without Regularization :", round(basic_val_acc, 4))
print("Dropout                :", round(dropout_val_acc, 4))
print("L2 Regularization      :", round(l2_val_acc, 4))

COMPARISON

Training Accuracy:
Without Regularization : 0.9998
Dropout                : 0.9923
L2 Regularization      : 0.9863

Validation Accuracy:
Without Regularization : 0.8794
Dropout                : 0.8774
L2 Regularization      : 0.8742
